In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

In [11]:
df_train = pd.read_csv(r"../data/selected_col/model_train.csv")

df_train = df_train.drop(columns=["Unnamed: 0.1",	"Unnamed: 0"])

df_train

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,dx_encode
0,HAM_0007001,ISIC_0026573,nv,histo,60.0,female,lower extremity,../data/all_image/ISIC_0026573.jpg,5
1,HAM_0006055,ISIC_0031456,nv,follow_up,75.0,female,lower extremity,../data/all_image/ISIC_0031456.jpg,5
2,HAM_0006068,ISIC_0024591,nv,follow_up,35.0,male,back,../data/all_image/ISIC_0024591.jpg,5
3,HAM_0001590,ISIC_0025722,nv,follow_up,50.0,male,lower extremity,../data/all_image/ISIC_0025722.jpg,5
4,HAM_0002015,ISIC_0034035,nv,histo,30.0,female,lower extremity,../data/all_image/ISIC_0034035.jpg,5
...,...,...,...,...,...,...,...,...,...
7004,HAM_0000639,ISIC_0025257,nv,histo,70.0,female,upper extremity,../data/all_image/ISIC_0025257.jpg,5
7005,HAM_0007043,ISIC_0029947,bkl,consensus,75.0,male,face,../data/all_image/ISIC_0029947.jpg,2
7006,HAM_0001568,ISIC_0030614,nv,follow_up,50.0,male,trunk,../data/all_image/ISIC_0030614.jpg,5
7007,HAM_0005800,ISIC_0028864,nv,follow_up,40.0,female,upper extremity,../data/all_image/ISIC_0028864.jpg,5


In [12]:
df_test = pd.read_csv(r"../data/selected_col/model_test.csv")
df_test = df_test.drop(columns=["Unnamed: 0.1",	"Unnamed: 0"])

df_test

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,dx_encode
0,HAM_0000627,ISIC_0034019,nv,consensus,50.0,unknown,unknown,../data/all_image/ISIC_0034019.jpg,5
1,HAM_0001761,ISIC_0026732,nv,follow_up,30.0,male,abdomen,../data/all_image/ISIC_0026732.jpg,5
2,HAM_0002726,ISIC_0025473,nv,consensus,5.0,male,foot,../data/all_image/ISIC_0025473.jpg,5
3,HAM_0003424,ISIC_0033552,nv,histo,45.0,male,back,../data/all_image/ISIC_0033552.jpg,5
4,HAM_0004081,ISIC_0031957,mel,histo,70.0,female,lower extremity,../data/all_image/ISIC_0031957.jpg,4
...,...,...,...,...,...,...,...,...,...
1498,HAM_0006808,ISIC_0027438,nv,follow_up,65.0,male,lower extremity,../data/all_image/ISIC_0027438.jpg,5
1499,HAM_0002425,ISIC_0032682,nv,consensus,20.0,male,back,../data/all_image/ISIC_0032682.jpg,5
1500,HAM_0003747,ISIC_0026582,nv,follow_up,50.0,male,trunk,../data/all_image/ISIC_0026582.jpg,5
1501,HAM_0002807,ISIC_0025050,nv,follow_up,45.0,male,trunk,../data/all_image/ISIC_0025050.jpg,5


In [13]:
df_val = pd.read_csv(r"../data/selected_col/model_test.csv")
df_val = df_val.drop(columns=["Unnamed: 0.1",	"Unnamed: 0"])

df_val

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,dx_encode
0,HAM_0000627,ISIC_0034019,nv,consensus,50.0,unknown,unknown,../data/all_image/ISIC_0034019.jpg,5
1,HAM_0001761,ISIC_0026732,nv,follow_up,30.0,male,abdomen,../data/all_image/ISIC_0026732.jpg,5
2,HAM_0002726,ISIC_0025473,nv,consensus,5.0,male,foot,../data/all_image/ISIC_0025473.jpg,5
3,HAM_0003424,ISIC_0033552,nv,histo,45.0,male,back,../data/all_image/ISIC_0033552.jpg,5
4,HAM_0004081,ISIC_0031957,mel,histo,70.0,female,lower extremity,../data/all_image/ISIC_0031957.jpg,4
...,...,...,...,...,...,...,...,...,...
1498,HAM_0006808,ISIC_0027438,nv,follow_up,65.0,male,lower extremity,../data/all_image/ISIC_0027438.jpg,5
1499,HAM_0002425,ISIC_0032682,nv,consensus,20.0,male,back,../data/all_image/ISIC_0032682.jpg,5
1500,HAM_0003747,ISIC_0026582,nv,follow_up,50.0,male,trunk,../data/all_image/ISIC_0026582.jpg,5
1501,HAM_0002807,ISIC_0025050,nv,follow_up,45.0,male,trunk,../data/all_image/ISIC_0025050.jpg,5


In [14]:
img_size = (256,256)
batch_size = 32

In [15]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(factor=0.15, value_range=(0.0, 1.0)),
    tf.keras.layers.RandomCrop(224,224)
])

In [16]:
def preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, img_size)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

In [17]:
def preprocess_train(image_path, label):
    image, label = preprocess_image(image_path, label)
    image = data_augmentation(image)
    return image, label

In [18]:
def preprocess_test(image_path, label):
    image, label = preprocess_image(image_path, label)
    return image, label

In [19]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset = (
    train_dataset
    .map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [20]:
val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset = (
    val_dataset
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


In [21]:
test_dataset = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset = (
    test_dataset
    .map(preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [25]:
y_train = df_train["dx_encode"].values


class_weights = compute_class_weight(class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train)

# Convert to dictionary
class_weight_dict = dict(enumerate(class_weights))

class_weight_dict

{0: np.float64(4.372426699937617),
 1: np.float64(2.7813492063492062),
 2: np.float64(1.3020620471855842),
 3: np.float64(12.51607142857143),
 4: np.float64(1.2853475151292866),
 5: np.float64(0.2133572798392743),
 6: np.float64(10.113997113997113)}